# Lot 46 — G5 Deep-Target Training & Strategic Qualification

Notebook Colab **orchestrateur mince**. La logique scientifique réside dans `packages/songo_ai/training/lot46.py` et `apps/trainer/scripts/run_srn_lot46.py`.

Ce notebook ne contourne aucun gate : si les résultats finaux Lot45, les checksums, les checkpoints G4R ou CUDA manquent, l'entraînement reste interdit. Il ne promeut jamais G5 automatiquement.

In [ ]:
# Configuration unique du run — modifier uniquement ces chemins.
from pathlib import Path

REPO = Path('/content/songo')
DRIVE_ROOT = Path('/content/drive/MyDrive/songo-ai')
LOT45_ARCHIVE = DRIVE_ROOT / 'exports/lot45_results.tar.gz'
LOT46_OUTPUT = DRIVE_ROOT / 'experiments/lot46_g5_training'
LOT46_EXPORT = DRIVE_ROOT / 'exports/lot46_results.tar.gz'
SEED = 20264601
print({'repo': str(REPO), 'lot45_archive': str(LOT45_ARCHIVE), 'output': str(LOT46_OUTPUT)})

In [ ]:
# Colab seulement : monter Drive.
from google.colab import drive
drive.mount('/content/drive')

## 1. Dépôt et environnement

Le dépôt doit déjà être cloné dans `REPO`. Aucun code scientifique n'est défini dans le notebook.

In [ ]:
import os, subprocess, sys, torch
assert REPO.is_dir(), f'Dépôt absent: {REPO}'
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[train,dev]'], check=True)
print({'torch': torch.__version__, 'cuda_available': torch.cuda.is_available(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})
assert torch.cuda.is_available(), 'CUDA indisponible: entraînement principal interdit; utiliser CPU uniquement pour audit/tests/smoke.'

## 2. Import transactionnel des résultats Lot45

L'archive est vérifiée avant extraction. L'archive d'entrée `lot45_inputs.tar.gz` n'est pas acceptée comme résultat.

In [ ]:
import hashlib, json, shutil, tarfile, tempfile

assert LOT45_ARCHIVE.is_file(), f'Résultat Lot45 absent: {LOT45_ARCHIVE}'
expected_sidecar = Path(str(LOT45_ARCHIVE) + '.sha256')
assert expected_sidecar.is_file(), f'Checksum absent: {expected_sidecar}'
actual = hashlib.sha256(LOT45_ARCHIVE.read_bytes()).hexdigest()
expected = expected_sidecar.read_text().split()[0]
assert actual == expected, f'Checksum Lot45 invalide: {actual} != {expected}'

destination = REPO / 'data/experiments/lot45_g5_target_generation'
with tempfile.TemporaryDirectory(dir='/content') as temporary:
    staging = Path(temporary)
    with tarfile.open(LOT45_ARCHIVE, 'r:gz') as archive:
        root = staging.resolve()
        for member in archive.getmembers():
            target = (staging / member.name).resolve()
            assert target == root or root in target.parents, f'Chemin dangereux dans archive: {member.name}'
        archive.extractall(staging)
    matches = list(staging.rglob('dataset/g5_deep_autonomous_reanalysis_v1.jsonl.gz'))
    assert len(matches) == 1, f'Dataset Lot45 final introuvable ou ambigu: {matches}'
    extracted_root = matches[0].parent.parent
    assert (extracted_root / 'decision.json').is_file() and (extracted_root / 'dataset_manifest.json').is_file()
    if destination.exists():
        shutil.rmtree(destination)
    shutil.copytree(extracted_root, destination)
print({'lot45_sha256': actual, 'extracted_to': str(destination)})

## 3. Tests puis audit/preflight

Le runner écrit tous les rapports dans un répertoire propre à l'expérience.

In [ ]:
env = {**os.environ, 'PYTHONPATH': f'{REPO / "packages"}:{REPO / "apps/trainer/scripts"}'}
subprocess.run([sys.executable, '-m', 'pytest', '-q', 'packages/songo_ai/tests/test_lot46_g5_training.py'], check=True, env=env)
subprocess.run([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'prepare', '--output', str(LOT46_OUTPUT)], check=True, env=env)
preflight = json.loads((LOT46_OUTPUT / 'preflight_report.json').read_text())
display(preflight)
assert preflight['SCIENTIFIC_TRAINING_ALLOWED'] == 'YES', preflight

## 4. Smoke end-to-end

Forward, losses Policy/Value masquées, backward, clipping, optimizer step et checkpoint atomique réel.

In [ ]:
subprocess.run([sys.executable, 'apps/trainer/scripts/run_srn_lot46.py', '--stage', 'smoke', '--output', str(LOT46_OUTPUT)], check=True, env=env)
smoke = json.loads((LOT46_OUTPUT / 'smoke_test_report.json').read_text())
display(smoke)
assert smoke['status'] == 'PASS', smoke

## 5. Entraînement, qualification et export

Les étapes longues ne doivent être exécutées que lorsque le runner Lot46 expose les stages `train`, `validate`, `arena`, `finalize` et `export`. Cette version s'arrête volontairement après le smoke : elle ne présente pas une implémentation partielle comme un entraînement scientifique opérationnel.

**État attendu actuellement : `IMPLEMENTATION_STATUS = PREFLIGHT_AND_SMOKE_ONLY`.**

In [ ]:
raise RuntimeError(
    'CONTROLLED STOP: les stages complets train/strategic-validation/arenas/finalize/export '
    'ne sont pas encore exposés par run_srn_lot46.py. Ne pas déclarer G5 entraîné ou promu.'
)